In [32]:
import json
import math
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2

GTFS_REALTIME_URL = "https://realtime.gtfs.de/realtime-free.pb"



In [33]:


GTFS_REALTIME_URL = "https://realtime.gtfs.de/realtime-free.pb"


def load_gtfs_realtime_feed(url=GTFS_REALTIME_URL):
    """
    Download and parse the current GTFS-RT feed.
    """

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    return feed


def preprocess_gtfs(
    data_dir,
    munich_geojson_path,
):
    """
    Preprocess static GTFS data and filter stops geographically
    to the Munich boundary.

    Parameters
    ----------
    data_dir : str
        Directory containing routes.txt, trips.txt and munich_stops.csv.

    munich_geojson_path : str
        Path to the GeoJSON file containing the Munich boundary.

    Returns
    -------
    trip_lines : dict
        Mapping from trip_id to line name.

    stop_names : dict
        Mapping from stop_id to stop name.
    """

    routes_df = pd.read_csv(
        f"{data_dir}/routes.txt",
        dtype={
            "route_id": str,
        },
    )

    trips_df = pd.read_csv(
        f"{data_dir}/trips.txt",
        dtype={
            "trip_id": str,
            "route_id": str,
        },
    )

    stops_df = pd.read_csv(
        f"{data_dir}/munich_stops.csv",
        dtype={
            "stop_id": str,
        },
    )

    route_lines = (
        routes_df
        .set_index("route_id")["route_short_name"]
        .to_dict()
    )

    trip_lines = (
        trips_df
        .set_index("trip_id")["route_id"]
        .map(route_lines)
        .dropna()
        .to_dict()
    )

    stop_names = (
        stops_df
        .set_index("stop_id")["stop_name"]
        .to_dict()
    )

    return trip_lines, stop_names


def parse_trip_updates(
    feed,
    stop_names,
    trip_lines,
    observation_timestamp,
):
    """
    Parse GTFS-RT trip updates into a pandas DataFrame.
    """

    rows = []

    for entity in feed.entity:

        if not entity.HasField("trip_update"):
            continue

        trip = entity.trip_update.trip

        line = trip_lines.get(str(trip.trip_id))

        if line is None:
            continue

        for stop in entity.trip_update.stop_time_update:

            stop_id = str(stop.stop_id)

            if stop_id not in stop_names:
                continue

            row = {
                "observation_timestamp": observation_timestamp,
                "trip_id": trip.trip_id,
                "start_date": trip.start_date,
                "line": line,
                "stop_id": stop_id,
                "stop_name": stop_names[stop_id],
                "stop_sequence": stop.stop_sequence,
            }

            if stop.HasField("departure"):
                row["departure_time"] = datetime.fromtimestamp(
                    stop.departure.time
                )
                row["departure_delay"] = stop.departure.delay

            if stop.HasField("arrival"):
                row["arrival_time"] = datetime.fromtimestamp(
                    stop.arrival.time
                )
                row["arrival_delay"] = stop.arrival.delay

            rows.append(row)

    return pd.DataFrame(rows)


def load_new_data(
    data_dir="../data",
    munich_geojson_path="munich.geojson",
):
    """
    Load and process the current MVV real-time data.
    """

    observation_timestamp = datetime.now()

    feed = load_gtfs_realtime_feed()

    trip_lines, stop_names = preprocess_gtfs(
        data_dir=data_dir,
        munich_geojson_path=munich_geojson_path,
    )

    realtime_df = parse_trip_updates(
        feed=feed,
        stop_names=stop_names,
        trip_lines=trip_lines,
        observation_timestamp=observation_timestamp,
    )

    return realtime_df

In [34]:
new_data = load_new_data()


In [44]:
new_data.tail(400)


,observation_timestamp,trip_id,start_date,line,stop_id,stop_name,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
343,2026-09-09 00:30:20.867694,898688,20260908,5021,327142,Thalkirchen (Tierpark),15,2026-09-09 00:10:48,-12.0,2026-09-09 00:10:48,-12.0
344,2026-09-09 00:30:20.867694,831686,20260909,142,370283,Klinikum Großhadern Ost,22,NaT,NaN,NaT,NaN
345,2026-09-09 00:30:20.867694,1298909,20260908,AST 226,272590,Rotkreuzplatz,13,2026-09-09 00:19:00,0.0,NaT,NaN
346,2026-09-09 00:30:20.867694,1283276,20260908,300,607758,Allach Bf. Ost,11,2026-09-08 23:47:00,60.0,2026-09-08 23:47:00,60.0
347,2026-09-09 00:30:20.867694,1320659,20260908,U1,249322,Erlbachstraße,25,NaT,NaN,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...
738,2026-09-09 00:30:20.867694,68190,20260908,976,302655,"Berg am Laim, München",8,2026-09-08 23:26:00,0.0,2026-09-08 23:26:00,0.0
739,2026-09-09 00:30:20.867694,1687930,20260909,M43,283304,Flemischweg,14,2026-09-09 00:52:00,0.0,2026-09-09 00:49:00,60.0
740,2026-09-09 00:30:20.867694,1244594,20260909,U76,370283,Klinikum Großhadern Ost,22,NaT,NaN,NaT,NaN
741,2026-09-09 00:30:20.867694,910941,20260909,RB36,370283,Klinikum Großhadern Ost,13,NaT,NaN,NaT,NaN


In [36]:
new_data.to_csv("../data/data_exploration.csv")